# 🕵️ Dia 47 — Metodologia de atribuição e a armadilha do ferramental compartilhado

**Investigador Cyber 90 Dias** · Fase 3 · Rank Jutsu `B`

> Baseado no *Day 47* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-3-cyber-threat-intelligence/day-47.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

O presidente da Serra Azul escreveu na ata: *"Quem é o Pescador? Queremos um nome."* E um post no LinkedIn decretou: *"mesmo painel de um kit vendido no Telegram, caso resolvido"*. Rafa: *"Somando três coisas que se compram, você não chega numa pessoa: chega numa loja."*

Neste notebook você vai:

1. **Transformar o fluxograma de graduação em função** (custo de falsificar + independência).
2. **Graduar o caso Turla** (o do original) e o **caso IscaKit**.
3. **Colapsar itens dependentes** e contar quantas evidências independentes sobram para cada ator.
4. **Gerar o mermaid do grafo de evidências** com caixa de dependência e painel restrito.
5. **Rodar a checagem da falácia** do ferramental compartilhado.
6. **Jogar o quiz "quanto custa falsificar?"**.

🐍 Só biblioteca padrão.

## 1. Os níveis de atribuição e a função de graduação

O fluxograma do passo 3 do original tem duas perguntas. Em código:

1. **Custo de falsificar**: baixo (público, comercial, vazado, strings, fuso, técnica comum), médio (exige acesso que outro ator controla: *pergunte se esse acesso foi obtido*), alto (difícil, **e você escreve por quê**).
2. **Mesmo ato?** Itens que nascem do mesmo ato contam como **um**.

In [ ]:
NIVEIS = {1: "cluster", 2: "grupo conhecido", 3: "patrocinador", 4: "indivíduos"}

# Palavras que, na descrição do item, indicam algo que se compra, se baixa ou se planta.
BAIXO = ("público", "comercial", "vendido", "vazado", "string", "comentário", "fuso", "horário",
         "idioma", "técnica comum", "print", "anúncio", "painel", "erros de português")
MEDIO = ("infraestrutura", "servidor", "implante", "certificado", "c2")

def graduar(item: dict) -> dict:
    """Aplica as duas perguntas do fluxograma. 'motivo_alto' é obrigatório para 'alto'."""
    texto = item["item"].lower()
    if item.get("custo"):                                   # o analista já decidiu: respeitamos
        custo = item["custo"]
    elif any(p in texto for p in BAIXO):
        custo = "baixo"
    elif any(p in texto for p in MEDIO):
        custo = "médio"
    else:
        custo = "indefinido"                                # a função não decide sozinha: você decide
    problemas = []
    if custo == "alto" and not item.get("motivo_alto"):
        problemas.append("graduado ALTO sem motivo escrito (checkpoint do original)")
    if custo == "indefinido":
        problemas.append("custo não classificado: decida e escreva o motivo")
    return {**item, "custo": custo, "problemas": problemas}

print("Função pronta. Níveis:", NIVEIS)

## 2. O caso Turla (outubro de 2019)

Os três primeiros itens são os do original. Os demais são **exemplos de partida** a conferir no [alerta conjunto](https://www.ncsc.gov.uk/news/turla-group-exploits-iran-apt-to-expand-coverage-of-victims) e completar até 8–12 itens: o original pede que você extraia os itens do documento, não do notebook.

In [ ]:
TURLA = [
    {"id": "E1", "item": "Implante Neuron ou Nautilus no host", "aponta_para": "Grupo iraniano",
     "classe": "Malware", "ato": "aquisicao-pelo-turla", "visivel": "respondedor", "custo": "médio",
     "nota": "Exige obter o implante e os dados de operação, o que o Turla fez"},
    {"id": "E2", "item": "Tráfego C2 para infraestrutura usada antes pelo grupo iraniano", "aponta_para": "Grupo iraniano",
     "classe": "Infraestrutura", "ato": "aquisicao-pelo-turla", "visivel": "respondedor", "custo": "médio",
     "nota": "Exige acesso à infraestrutura do outro grupo, que o Turla tinha"},
    {"id": "E3", "item": "Vítima é organização governamental no Oriente Médio", "aponta_para": "ninguém em particular",
     "classe": "Vitimologia", "ato": "escolha-de-alvo", "visivel": "respondedor", "custo": "baixo",
     "nota": "Os dois grupos já miraram a região: circunstancial"},
    {"id": "E4", "item": "Ferramentas de administração e técnicas comuns na rede da vítima (a conferir no alerta)",
     "aponta_para": "ninguém em particular", "classe": "Tradecraft", "ato": "operacao", "visivel": "respondedor",
     "custo": "baixo", "nota": "Técnica comum: compartilhada por muitos grupos (Dia 38)"},
    {"id": "G1", "item": "Como o Turla adquiriu as ferramentas e o acesso", "aponta_para": "Turla",
     "classe": "Fontes externas", "ato": "inteligencia-governamental", "visivel": "governo", "custo": "alto",
     "motivo_alto": "Fontes e métodos de governo, não reproduzíveis por quem planta pistas técnicas"},
]
TURLA = [graduar(i) for i in TURLA]
for i in TURLA:
    print(f"{i['id']:<3} [{i['custo']:<5}] {i['aponta_para']:<22} {i['item'][:60]}"
          + ("   ⚠️ " + "; ".join(i["problemas"]) if i["problemas"] else ""))

## 3. Colapsar dependências e contar o que sobra

Dois itens do mesmo ato valem **um**. A função abaixo agrupa por `ato`, conta evidências **independentes** por ator, e faz isso duas vezes: com tudo, e só com o que o **respondedor** via. A diferença entre as duas contagens é o tamanho da visão retrospectiva que o checkpoint pede para você vigiar.

In [ ]:
from collections import defaultdict

def independentes(itens, so_visiveis_a=None):
    """{ator: [atos distintos]} — cada ato conta uma vez, não importa quantos itens gerou."""
    por_ator = defaultdict(set)
    for i in itens:
        if so_visiveis_a and i["visivel"] != so_visiveis_a:
            continue
        por_ator[i["aponta_para"]].add(i["ato"])
    return {a: sorted(atos) for a, atos in por_ator.items()}

def relatorio_independencia(itens, visivel):
    brutos = defaultdict(int)
    for i in itens:
        if i["visivel"] == visivel:
            brutos[i["aponta_para"]] += 1
    print(f"Só o que '{visivel}' via:")
    for ator, atos in independentes(itens, visivel).items():
        print(f"   {ator:<24} itens={brutos[ator]}  →  independentes={len(atos)}  {atos}")
    print("Com tudo (inclusive o que só outros viam):")
    for ator, atos in independentes(itens).items():
        print(f"   {ator:<24} independentes={len(atos)}")

relatorio_independencia(TURLA, "respondedor")

> 💡 Para o respondedor de 2019, o "Grupo iraniano" tinha **dois itens, mas um ato só**. E nada apontava para o Turla. Um contrafactual honesto conclui pelo ator errado, ou que a evidência não decidia, exatamente como o checkpoint pede.

## 4. ➕ O caso IscaKit

Dados de `caso-continuo/dados/dia-47/evidencias-iscakit.csv` (fictícios), com plano B embutido.

In [ ]:
import csv
import io
import urllib.request

URL = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/"
       "main/caso-continuo/dados/dia-47/evidencias-iscakit.csv")
EMBUTIDO = """id,item,aponta_para,classe_odni,ato_de_origem,visivel_para
I1,Painel de administração do IscaKit com o mesmo layout de um kit vendido em 2024,vendedor do kit,malware,compra-do-kit,Vigia
I2,Comentário '// pescador' no JavaScript das páginas,vendedor do kit,malware,compra-do-kit,Vigia
I3,Caminho /api/recebe para envio das credenciais,vendedor do kit,malware,compra-do-kit,Vigia
I4,Anúncio do 'IscaKit cooperativas' em print de grupo de Telegram (não verificado),vendedor do kit,fonte externa,anuncio,Vigia
I5,Certificado TLS com SAN compartilhado entre serraazul-pix.example e serraazul-seguranca.example,cluster CA-ISCA-01,infraestrutura,emissao-cert-1,Vigia
I6,Mesmo servidor 203.0.113.77 para as ondas 1 a 3,cluster CA-ISCA-01,infraestrutura,contratacao-hospedagem-1,Vigia
I7,Disparos sempre entre 07h00 e 09h00 BRT,cluster CA-ISCA-01,tradecraft,agendamento-sms,Vigia
I8,Alvos exclusivamente cooperativas de crédito de Minas Gerais,ninguém em particular,intenção,escolha-de-alvo,Vigia
I9,Erros de português no SMS das ondas 4 e 5,ninguém em particular,tradecraft,redacao-sms-2,Vigia
I10,Dados cadastrais do registrante dos domínios,pessoa (não avaliado),fonte externa,registro-dominio,só com ordem judicial
I11,Registros de conexão do provedor de hospedagem,pessoa (não avaliado),fonte externa,acesso-servidor,só com ordem judicial
I12,Titularidade das contas que receberam os Pix,pessoa (não avaliado),fonte externa,recebimento-pix,só com ordem judicial
"""
try:
    with urllib.request.urlopen(URL, timeout=10) as r:
        texto = r.read().decode("utf-8")
except Exception as erro:
    texto = EMBUTIDO
    print(f"Sem rede ({type(erro).__name__}): usando a cópia embutida.")

ISCA = []
for l in csv.DictReader(io.StringIO(texto)):
    ISCA.append(graduar({"id": l["id"], "item": l["item"], "aponta_para": l["aponta_para"],
                         "classe": l["classe_odni"], "ato": l["ato_de_origem"],
                         "visivel": l["visivel_para"],
                         # Itens restritos: não somos nós que graduamos; ficam fora da nossa análise.
                         "custo": "não avaliado" if l["visivel_para"] != "Vigia" else ""}))

for i in ISCA:
    print(f"{i['id']:<4} [{i['custo']:<12}] {i['aponta_para']:<22} {i['item'][:58]}"
          + ("   ⚠️ " + "; ".join(i["problemas"]) if i["problemas"] else ""))

Repare nos `⚠️ custo não classificado`: a função **não sabe** decidir sobre "certificado com SAN compartilhado" ou "disparos entre 07h e 09h"? Sabe, para alguns (palavras-chave), mas o ponto é outro: **você** decide e escreve o motivo. Use o formulário.

In [ ]:
#@title Suas decisões de custo (IscaKit) { display-mode: "form" }
custo_I3 = "baixo"  #@param ["baixo", "médio", "alto"]
motivo_I3 = "Vem pronto no kit: todo comprador recebe o mesmo caminho"  #@param {type:"string"}
custo_I5 = "médio"  #@param ["baixo", "médio", "alto"]
motivo_I5 = "Exige controlar a emissão do certificado; outro ator só faria apontar para cá se controlasse os dois domínios"  #@param {type:"string"}
custo_I6 = "baixo"  #@param ["baixo", "médio", "alto"]
motivo_I6 = "Hospedagem compartilhada e alugável por qualquer um"  #@param {type:"string"}
custo_I7 = "baixo"  #@param ["baixo", "médio", "alto"]
motivo_I7 = "Horário de agendamento é trivial de escolher ou imitar"  #@param {type:"string"}
custo_I8 = "baixo"  #@param ["baixo", "médio", "alto"]
motivo_I8 = "Muitos golpistas miram cooperativas; circunstancial"  #@param {type:"string"}

for iid, custo, motivo in (("I3", custo_I3, motivo_I3), ("I5", custo_I5, motivo_I5), ("I6", custo_I6, motivo_I6),
                           ("I7", custo_I7, motivo_I7), ("I8", custo_I8, motivo_I8)):
    item = next(i for i in ISCA if i["id"] == iid)
    item.update(graduar({**item, "custo": custo, "motivo_alto": motivo if custo == "alto" else "",
                         "nota": motivo}))
pendentes = [i["id"] for i in ISCA if i["problemas"]]
print("Itens ainda com problema:", pendentes or "nenhum ✅")
print()
relatorio_independencia(ISCA, "Vigia")

### O nível de atribuição defensável

A regra (síntese do Conceito, não do original): para subir de nível, é preciso **mais de um ato independente** apontando para o mesmo candidato, com pelo menos um item que não seja de custo baixo. Veja o que a evidência visível à Vigia permite.

In [ ]:
def nivel_defensavel(itens, visivel):
    atos = independentes(itens, visivel)
    custo_ato = defaultdict(set)
    for i in itens:
        if i["visivel"] == visivel:
            custo_ato[(i["aponta_para"], i["ato"])].add(i["custo"])
    for ator, lista in atos.items():
        fortes = [a for a in lista if custo_ato[(ator, a)] & {"médio", "alto"}]   # "indefinido" não conta
        veredito = "sustenta ligação" if len(lista) >= 2 and fortes else "NÃO sustenta ligação"
        print(f"   {ator:<24} atos={len(lista)}  com custo > baixo={len(fortes)}  → {veredito}")

print("IscaKit, visão da Vigia:")
nivel_defensavel(ISCA, "Vigia")
print("\n➡️  'cluster CA-ISCA-01' é o agrupamento que nós DEFINIMOS (nível 1).")
print("➡️  'vendedor do kit' tem 4 itens, mas só atos de baixo custo (comprar, anunciar): o post do LinkedIn não sobe ao nível 2.")
print("➡️  Itens 'só com ordem judicial' (I10–I12) são o caminho para o nível 4. Ficam com a polícia.")

## 5. O grafo de evidências, em mermaid

A célula gera o diagrama no formato do esqueleto do original: atores escuros, itens coloridos por classe, caixa tracejada para cada grupo de itens do **mesmo ato**, e o painel restrito separado. Cole o resultado no [mermaid.live](https://mermaid.live/) ou num README do GitHub, e depois redesenhe no diagrams.net como o original pede.

In [ ]:
CORES = {"malware": "#dae8fc", "infraestrutura": "#d5e8d4", "infrastructure": "#d5e8d4",
         "vitimologia": "#fff2cc", "intenção": "#fff2cc", "tradecraft": "#ffe6cc",
         "fonte externa": "#e1d5e7", "fontes externas": "#e1d5e7"}

def mermaid(itens, visivel, titulo_restrito):
    linhas = ["graph LR"]
    atores = sorted({i["aponta_para"] for i in itens})
    aid = {a: f"A{n}" for n, a in enumerate(atores)}
    for a in atores:
        linhas.append(f'    {aid[a]}["{a}"]')
    por_ato = defaultdict(list)
    for i in itens:
        por_ato[(i["visivel"], i["ato"])].append(i)
    restritos = []
    for (vis, ato), grupo in por_ato.items():
        nos = [f'        {i["id"]}["{i["id"]} {i["item"][:38]}<br/>classe: {i["classe"]}"]' for i in grupo]
        if vis != visivel:
            restritos += nos
        elif len(grupo) > 1:                               # mesmo ato → caixa tracejada
            linhas.append(f'    subgraph {ato.replace("-", "_")}["Não independentes: {ato}"]')
            linhas += nos
            linhas.append("    end")
            linhas.append(f'    style {ato.replace("-", "_")} stroke-dasharray:5 5')
        else:
            linhas += ["    " + n.strip() for n in nos]
    if restritos:
        linhas.append(f'    subgraph RESTRITO["{titulo_restrito}"]')
        linhas += restritos
        linhas.append("    end")
    for i in itens:
        # custo baixo = aresta pontilhada (circunstancial); demais = contínua, rotulada com o custo
        if i["custo"] == "baixo":
            linhas.append(f'    {i["id"]} -. "baixo" .-> {aid[i["aponta_para"]]}')
        else:
            linhas.append(f'    {i["id"]} -- "{i["custo"]}" --> {aid[i["aponta_para"]]}')
    for i in itens:
        cor = CORES.get(i["classe"].lower(), "#f5f5f5")
        linhas.append(f'    style {i["id"]} fill:{cor},color:#000000')
    for a in atores:
        linhas.append(f'    style {aid[a]} fill:#333333,color:#ffffff')
    return "\n".join(linhas)

MMD_TURLA = mermaid(TURLA, "respondedor", "Visível só a investigadores do governo")
MMD_ISCA = mermaid(ISCA, "Vigia", "Visível só com ordem judicial (polícia)")
with open("grafo-turla.mmd", "w", encoding="utf-8") as f:
    f.write(MMD_TURLA + "\n")
with open("grafo-iscakit.mmd", "w", encoding="utf-8") as f:
    f.write(MMD_ISCA + "\n")
print(MMD_ISCA)
print("\n💾 grafo-turla.mmd e grafo-iscakit.mmd")

## 6. A checagem da falácia (passo 6)

Quantos grupos do ATT&CK usam cada ferramenta? As cinco contagens do Conceito (ATT&CK v19.2, relacionamentos diretos) vêm embutidas. Se você ligar `BAIXAR_ATTACK`, a célula baixa o *bundle* Enterprise do repositório oficial `mitre-attack/attack-stix-data` (dezenas de MB) e recalcula na versão atual.

Depois, cole a lista de ferramentas da tabela "Ferramental" do seu Dia 37.

In [ ]:
#@title Checagem da falácia { display-mode: "form" }
BAIXAR_ATTACK = False  #@param {type:"boolean"}
MINHAS_FERRAMENTAS = "Mimikatz, PsExec, Cobalt Strike, PlugX, AdFind"  #@param {type:"string"}

import json
CONTAGEM_V19_2 = {"Mimikatz": 51, "PsExec": 38, "Cobalt Strike": 30, "PlugX": 15, "AdFind": 12}
contagem, versao = dict(CONTAGEM_V19_2), "ATT&CK v19.2 (do original)"

if BAIXAR_ATTACK:
    url = "https://raw.githubusercontent.com/mitre-attack/attack-stix-data/master/enterprise-attack/enterprise-attack.json"
    try:
        with urllib.request.urlopen(url, timeout=120) as r:
            bundle = json.load(r)
        vivos = {o["id"]: o for o in bundle["objects"] if not o.get("revoked") and not o.get("x_mitre_deprecated")}
        nome = {i: o.get("name") for i, o in vivos.items()}
        grupos = defaultdict(set)
        for o in bundle["objects"]:
            if (o.get("relationship_type") == "uses" and o.get("source_ref", "").startswith("intrusion-set--")
                    and o.get("target_ref", "").split("--")[0] in ("malware", "tool")
                    and o["source_ref"] in vivos and o["target_ref"] in vivos and not o.get("revoked")):
                grupos[nome[o["target_ref"]]].add(o["source_ref"])
        contagem = {k: len(v) for k, v in grupos.items()}
        colecao = next((o for o in bundle["objects"] if o["type"] == "x-mitre-collection"), {})
        versao = f"ATT&CK {colecao.get('x_mitre_version', '(versão atual)')} (baixado agora)"
    except Exception as erro:
        print(f"Não deu para baixar ({type(erro).__name__}); usando as contagens do original.")

print("Fonte das contagens:", versao)
for f in [x.strip() for x in MINHAS_FERRAMENTAS.split(",") if x.strip()]:
    n = contagem.get(f)
    rotulo = "sem dado: confira no ATT&CK" if n is None else ("NÃO IDENTIFICADORA (>10 grupos)" if n > 10 else "pode ajudar, com cautela")
    print(f"   {f:<16} grupos={n!s:<4} → {rotulo}")

## 7. 🎮 Quiz: quanto custa falsificar?

Responda **1** baixo · **2** médio · **3** alto.

In [ ]:
QUIZ = [
    ("Fuso horário UTC+3 nos horários de compilação do malware.", "1", "Trivial de alterar ou plantar."),
    ("Uso de Mimikatz na rede da vítima.", "1", "Ligado a dezenas de grupos no ATT&CK: não identifica ninguém."),
    ("C2 em servidores que outro grupo usava antes.", "2", "Exige acesso que o outro grupo controla. Pergunte: esse acesso pode ter sido obtido? (Turla.)"),
    ("Comentário '// pescador' no código de um kit vendido.", "1", "Vem com o kit para qualquer comprador."),
    ("Rich header de um executável igual ao de malware de outro grupo, mas incoerente com o resto do build.", "1",
     "Copiável, e a incoerência indica que foi copiado (é o caso de amanhã)."),
    ("Registros de conexão do provedor, obtidos por ordem judicial, ligando o painel a uma conexão específica.", "3",
     "Alto, com motivo: vem de terceiro neutro, sob ordem judicial e cadeia de custódia. E não é a Vigia quem obtém."),
]
CUSTOS = {"1": "baixo", "2": "médio", "3": "alto"}
MODO_JOGO = True
acertos = 0
for n, (q, certo, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{n}/{len(QUIZ)}] {q}")
    if MODO_JOGO:
        r = ""
        while r not in CUSTOS:
            r = input("   1 baixo · 2 médio · 3 alto: ").strip()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌")
    print(f"   Gabarito: {CUSTOS[certo]} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")

## 8. 📝 Os dois contrafactuais

In [ ]:
#@title Contrafactuais { display-mode: "form" }
contrafactual_turla = "Com só o que o respondedor via, a evidência apontava para o grupo iraniano: implante e C2 coincidiam, mas vinham de um único ato de aquisição. Um analista cuidadoso teria avaliado como possibilidade realista a ligação ao grupo iraniano. Confiança: baixa. A conclusão correta (Turla) dependeu de fontes externas de governo, invisíveis ao respondedor."  #@param {type:"string"}
contrafactual_isca = "Com só o que a Vigia vê, o nível defensável é 1: o cluster CA-ISCA-01, definido por certificado e infraestrutura compartilhados. Os itens que apontam para o vendedor do kit nascem do mesmo ato de compra e são de baixo custo de falsificar; não sustentam ligação a um grupo. Quem pode chegar a pessoas é a polícia, com ordem judicial."  #@param {type:"string"}

import re
for nome, txt in (("Turla", contrafactual_turla), ("IscaKit", contrafactual_isca)):
    nomes_proprios = re.findall(r"\b[A-Z][a-z]+ [A-Z][a-z]+\b", txt)   # heurística: "Nome Sobrenome"
    print(f"{nome}: {'⚠️ possível nome de pessoa: ' + str(nomes_proprios) if nomes_proprios else 'sem nomes de pessoa ✅'}")
from pathlib import Path
Path("notas").mkdir(exist_ok=True)
Path("notas/dia47-contrafactuais.md").write_text(
    f"# Contrafactuais — Dia 47\n\n## Turla\n{contrafactual_turla}\n\n## IscaKit\n{contrafactual_isca}\n", encoding="utf-8")
print("💾 notas/dia47-contrafactuais.md")
try:
    from google.colab import files
    for n in ("notas/dia47-contrafactuais.md", "grafo-turla.mmd", "grafo-iscakit.mmd"):
        files.download(n)
except ImportError:
    pass

## ✅ Antes de fechar o Dia 47

- [ ] Nenhum item graduado "alto" sem motivo escrito
- [ ] Pelo menos um par de itens que pareciam independentes, colapsado num ato só
- [ ] Contrafactual do Turla que conclui pelo ator errado ou por "não decide", e explica por quê
- [ ] Contrafactual do IscaKit parado no nível 1
- [ ] (Original) 8–12 itens do alerta, grafo no diagrams.net, tabelas dos Dias 37 e 38 com contagens

**Amanhã (Dia 48):** falsas bandeiras. O caso Olympic Destroyer mostra uma pista plantada de propósito, e você escreve três declarações de atribuição, uma delas para o cluster CA-ISCA-01. É o fechamento da Fase 3.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>